# **AI TECH INSTITUTE** · *Intermediate AI & Data Science*
### PCA for Image Compression
**Instructor:** Amir Charkhi | **Focus:** Practical Dimensionality Reduction

---

## 🎯 What You'll Learn

- How images are represented as matrices
- How PCA compresses information
- Trade-off: Quality vs Compression
- Visual demonstration of reconstruction

---

## 💡 The Big Idea

```
Original Image: 1000 × 1000 = 1,000,000 values
                    ↓
              Apply PCA
                    ↓
Compressed:   Keep only 50 components = ~50,000 values
                    ↓
              Reconstruct
                    ↓
Result:       95% quality, 5% storage!
```

---

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

---
## 2. Understanding Images as Data

### 📷 How Computers See Images

```
Grayscale Image = 2D Matrix of pixel values

Example (8×8 image):
┌─────────────────────────────┐
│ 120  125  130  128  ...     │  ← Row 1
│ 122  127  132  130  ...     │  ← Row 2
│ 118  123  128  126  ...     │  ← Row 3
│ ...  ...  ...  ...  ...     │
└─────────────────────────────┘

Each value: 0 (black) to 255 (white)
```

---

In [ ]:
# Load sample face images from sklearn
from sklearn.datasets import fetch_olivetti_faces

faces = fetch_olivetti_faces()

In [ ]:
# Get the images and their shape
images = faces.images
images.shape

💡 **400 images, each 64×64 pixels**

In [ ]:
# Select one face for demonstration
sample_image = images[0]
sample_image.shape

In [ ]:
# Visualize the sample image
fig = px.imshow(
    sample_image,
    color_continuous_scale='gray',
    title='Original Image (64×64 = 4,096 values)',
    width=500,
    height=500
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Total storage needed
total_values = sample_image.shape[0] * sample_image.shape[1]
f"Storage: {total_values:,} values per image"

---
## 3. How PCA Compresses Images

### 📖 The Concept

```
Step 1: Treat each ROW of image as a data point
        64 rows × 64 columns = 64 samples with 64 features

Step 2: Find principal components
        PC1: Direction of most variance
        PC2: Second most variance
        ...

Step 3: Keep only top K components
        Instead of 64 values per row → K values

Step 4: Reconstruct using only K components
        Some detail lost, but major features preserved
```

---

### 🎯 Compression Ratio

```
Original:    64 × 64 = 4,096 values

With K=10:   Store:
             - 64 × 10 (transformed data) = 640
             - 10 × 64 (components) = 640
             Total: 1,280 values

Compression: 4,096 → 1,280 = 69% reduction!
```

---

## 4. Compress Single Image with Different Components

In [ ]:
def compress_image(image, n_components):
    """Compress image using PCA with n_components"""
    pca = PCA(n_components=n_components)
    
    # Transform (compress)
    compressed = pca.fit_transform(image)
    
    # Reconstruct
    reconstructed = pca.inverse_transform(compressed)
    
    # Calculate variance retained
    variance_retained = pca.explained_variance_ratio_.sum()
    
    return reconstructed, variance_retained

In [ ]:
# Test different compression levels
n_components_list = [1, 5, 10, 20, 32, 64]
reconstructions = {}

In [ ]:
for n in n_components_list:
    recon, var = compress_image(sample_image, n)
    reconstructions[n] = {'image': recon, 'variance': var}

### 📊 Visual Comparison

In [ ]:
# Create comparison grid
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, n in enumerate(n_components_list):
    img = reconstructions[n]['image']
    var = reconstructions[n]['variance']
    
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'K={n} ({var:.1%} variance)', fontsize=12)
    axes[i].axis('off')

plt.suptitle('Image Compression: Effect of Number of Components', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

💡 **Notice:**
- K=1: Just captures average brightness
- K=5: Basic face shape emerges
- K=10: Recognizable face
- K=20: Good quality
- K=32: Nearly perfect (50% compression!)
- K=64: Original (no compression)

---

## 5. Compression vs Quality Trade-off

In [ ]:
# Calculate metrics for all component levels
all_k = list(range(1, 65))
variances = []
compression_ratios = []

In [ ]:
for k in all_k:
    pca = PCA(n_components=k)
    pca.fit(sample_image)
    variances.append(pca.explained_variance_ratio_.sum())
    
    # Compression ratio: original / compressed size
    original = 64 * 64
    compressed = (64 * k) + (k * 64)  # transformed + components
    compression_ratios.append(original / compressed)

In [ ]:
# Create trade-off visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Quality vs Components', 'Compression Ratio vs Quality')
)

# Quality vs Components
fig.add_trace(
    go.Scatter(
        x=all_k, y=variances,
        mode='lines',
        line=dict(color='steelblue', width=3),
        name='Variance Retained'
    ),
    row=1, col=1
)

# Add threshold lines
fig.add_hline(y=0.95, line_dash='dash', line_color='green', row=1, col=1,
              annotation_text='95% quality')
fig.add_hline(y=0.90, line_dash='dash', line_color='orange', row=1, col=1,
              annotation_text='90% quality')

# Compression vs Quality
fig.add_trace(
    go.Scatter(
        x=variances, y=compression_ratios,
        mode='lines',
        line=dict(color='green', width=3),
        name='Compression'
    ),
    row=1, col=2
)

fig.update_xaxes(title_text='Number of Components (K)', row=1, col=1)
fig.update_yaxes(title_text='Variance Retained', row=1, col=1)
fig.update_xaxes(title_text='Quality (Variance Retained)', row=1, col=2)
fig.update_yaxes(title_text='Compression Ratio', row=1, col=2)

fig.update_layout(
    title='The Trade-off: Quality vs Compression',
    height=450, width=1000,
    template='plotly_white',
    showlegend=False
)
fig.show()

### 📋 Optimal Points

In [ ]:
# Find K needed for different quality levels
quality_targets = [0.90, 0.95, 0.99]
results = []

for target in quality_targets:
    k_needed = np.argmax(np.array(variances) >= target) + 1
    compression = compression_ratios[k_needed - 1]
    storage_pct = (1 / compression) * 100
    
    results.append({
        'Quality': f'{target:.0%}',
        'Components (K)': k_needed,
        'Compression Ratio': f'{compression:.1f}x',
        'Storage Used': f'{storage_pct:.1f}%'
    })

pd.DataFrame(results)

💡 **Key Insight:** With just 20 components, we get 95% quality using only ~31% of original storage!

---

## 6. What PCA Learns: Eigenfaces

### 📖 Principal Components = "Eigenfaces"

```
When we apply PCA to faces:

PC1: Most common variation (lighting?)
PC2: Second most common (pose?)
PC3: Third most common (expression?)
...

These components look like ghostly faces!
Called "eigenfaces" in facial recognition.
```

---

In [ ]:
# Get all faces as flattened vectors
X_faces = faces.data  # Already flattened: 400 faces × 4096 pixels

In [ ]:
# Fit PCA on all faces
pca_faces = PCA(n_components=50)
pca_faces.fit(X_faces)

In [ ]:
# Visualize first 10 eigenfaces
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

for i in range(10):
    eigenface = pca_faces.components_[i].reshape(64, 64)
    axes[i].imshow(eigenface, cmap='gray')
    axes[i].set_title(f'PC{i+1} ({pca_faces.explained_variance_ratio_[i]:.1%})', fontsize=10)
    axes[i].axis('off')

plt.suptitle('Eigenfaces: What PCA Learns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

💡 **Interpretation:**
- PC1: Overall brightness/contrast
- PC2-3: Lighting direction
- PC4+: Facial features (eyes, nose, mouth)

**Any face can be reconstructed as a weighted sum of these eigenfaces!**

---

## 7. Compress Multiple Faces

In [ ]:
# Select 5 different faces
face_indices = [0, 10, 20, 30, 40]
sample_faces = images[face_indices]

In [ ]:
# Compress with different K values
k_values = [5, 15, 30]

fig, axes = plt.subplots(5, 4, figsize=(12, 15))

for i, face in enumerate(sample_faces):
    # Original
    axes[i, 0].imshow(face, cmap='gray')
    axes[i, 0].set_title('Original' if i == 0 else '', fontsize=10)
    axes[i, 0].axis('off')
    
    # Compressed versions
    for j, k in enumerate(k_values):
        recon, var = compress_image(face, k)
        axes[i, j+1].imshow(recon, cmap='gray')
        axes[i, j+1].set_title(f'K={k} ({var:.0%})' if i == 0 else '', fontsize=10)
        axes[i, j+1].axis('off')

plt.suptitle('Image Compression Across Different Faces', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

💡 **PCA compression works consistently across different faces!**

---

## 8. Interactive: Build Your Own Compression

In [ ]:
# Reconstruct using specific number of components
def show_reconstruction(n_components):
    """Show original vs reconstructed with n_components"""
    recon, var = compress_image(sample_image, n_components)
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    # Original
    axes[0].imshow(sample_image, cmap='gray')
    axes[0].set_title('Original', fontsize=12)
    axes[0].axis('off')
    
    # Reconstructed
    axes[1].imshow(recon, cmap='gray')
    axes[1].set_title(f'Reconstructed (K={n_components}, {var:.1%} quality)', fontsize=12)
    axes[1].axis('off')
    
    # Difference
    diff = np.abs(sample_image - recon)
    axes[2].imshow(diff, cmap='hot')
    axes[2].set_title('Lost Information', fontsize=12)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Try K=10
show_reconstruction(10)

In [ ]:
# Try K=25
show_reconstruction(25)

In [ ]:
# Try K=45
show_reconstruction(45)

---
## 9. Real-World Application: Batch Compression

In [ ]:
# Compress all 400 faces using 30 components
n_components = 30

pca_batch = PCA(n_components=n_components)
X_compressed = pca_batch.fit_transform(X_faces)

In [ ]:
# Storage comparison
original_size = X_faces.shape[0] * X_faces.shape[1]
compressed_size = X_compressed.shape[0] * X_compressed.shape[1] + pca_batch.components_.size

pd.DataFrame([{
    'Original Size': f'{original_size:,} values',
    'Compressed Size': f'{compressed_size:,} values',
    'Compression': f'{original_size/compressed_size:.1f}x',
    'Quality': f'{pca_batch.explained_variance_ratio_.sum():.1%}'
}])

In [ ]:
# Reconstruct all faces
X_reconstructed = pca_batch.inverse_transform(X_compressed)

In [ ]:
# Show some reconstructions
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for i in range(10):
    ax = axes[i // 5, i % 5]
    recon = X_reconstructed[i * 40].reshape(64, 64)
    ax.imshow(recon, cmap='gray')
    ax.axis('off')

plt.suptitle(f'Batch Reconstruction: 400 Faces with K={n_components}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Summary: PCA Image Compression

### ✅ How It Works

```
1. LEARN PATTERNS
   PCA finds principal components (eigenfaces)
   These capture common variations

2. COMPRESS
   Transform image → K coefficients
   K << original dimensions

3. STORE
   Keep: K coefficients + K components
   Much smaller than original!

4. RECONSTRUCT
   Multiply coefficients × components
   Get back (approximate) image
```

---

### 📊 Key Numbers

| Components | Quality | Compression | Use Case |
|------------|---------|-------------|----------|
| 5 | ~60% | 6x | Thumbnails |
| 15 | ~85% | 2x | Preview |
| 30 | ~95% | 1.3x | Good quality |
| 50 | ~99% | 1.1x | Near-perfect |

---

### 💡 When to Use PCA Compression

**Good For:**
- ✅ Similar images (faces, documents)
- ✅ Preprocessing before ML
- ✅ Quick compression
- ✅ Noise reduction

**Not Ideal For:**
- ❌ Very diverse images
- ❌ Fine detail preservation
- ❌ Lossy-free requirements

**Real Applications:**
- Face recognition preprocessing
- Medical image analysis
- Satellite imagery
- Video compression
- Noise reduction in images

---

### 🔑 Key Takeaways

1. **Images are just matrices** - PCA treats rows as samples

2. **PCA finds patterns** - Principal components capture common structures

3. **Trade-off exists** - More components = better quality, less compression

4. **Sweet spot** - Often 20-30 components for 90%+ quality

5. **Eigenfaces** - Components look like ghostly face templates

6. **Batch processing** - Same components work for similar images

---

**You now understand how PCA compresses images!** 📷🗜️

---

**AI Tech Institute** | *Building Tomorrow's AI Engineers Today*